# 09_客服_案件處理統計

分類統計與平均的分母

任務：按問題類別計算案件數。；只對已結案案件算平均處理分鐘。；計算整體結案率，說明為何未結案不能當作 0 分鐘。

規則：每列一個案件；未結案的處理分鐘空白；平均耗時的分母只含已結案案件。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('09_客服_案件處理統計.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '09_客服_案件處理統計'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
from collections import Counter
import numpy as np
counts = Counter(r['問題類別'] for r in rows)
closed = [r for r in rows if r['狀態'] == '已結案']
mean = np.mean([r['處理分鐘'] for r in closed])
print('類別件數：', counts, '平均分鐘：', mean, '結案率：', f'{len(closed)/len(rows):.1%}')
assert dict(counts) == {'配送':3, '退貨':2, '商品':1}
assert mean == 42 and len(closed) == 5
wb.close()

## 驗收答案

配送 3 件、退貨 2 件、商品 1 件；已結案平均 42 分鐘，結案率 83.3%。